In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Italy Earthquake Catalog — Pre-Analysis (INGV 1985-2025)

Covers: catalog EDA, Gutenberg-Richter law, Omori-Utsu law.
Run as a script  : python ITALY_preanalysis.py
Convert to notebook: python convert_to_notebook.py ITALY_preanalysis.py notebooks/ITALY_preanalysis.ipynb
"""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.seismology import fit_gr_law, fit_omori
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR = Path("data/INGV")
CUT_YEAR = 1985
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "preanalysis")

## Data Loading and Basic Statistics

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df["year"] = df["time"].dt.year
df_net = (
    df[df["year"] >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['year'].min()}–{df_net['year'].max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Columns: {list(df_net.columns)}")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

_ITALY_POLYGON = Polygon([
    (13.5, 44.5),  # Northeast (Adriatic Sea)
    (19.0, 41.2),  # Southeast (Apulia Sea)
    (19.0, 35.6),  # Sea between Italy and Libya
    (11.7, 35.5),  # Sea east of Tunisia
    (11.6, 37.9),  # West of Sicily
    (7.2,  38.0),  # Southwest of Sardinia
    (7.2,  42.6),  # West of Corsica
    (5.2,  45.5),  # Lyon
    (11.6, 48.4),  # Munich
    (16.0, 46.7),  # Between Austria and Slovenia
    (13.5, 44.5),  # closing vertex
])
df_net = df_net[df_net.apply(
    lambda r: _ITALY_POLYGON.contains(Point(r["longitude"], r["latitude"])), axis=1
)].reset_index(drop=True)
print(f"After polygon filter: {len(df_net):,} earthquakes remain")

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)}")

## Temporal Distribution

Annual event counts reveal catalog completeness and major seismic episodes.
The 1985 cut year is set to avoid the sharp improvement in detection sensitivity
in earlier decades; year-on-year stability of rates is the key quality indicator.

A sudden spike in a given year (e.g., 2016 Amatrice, 2009 L'Aquila) reflects
mainshock–aftershock sequences rather than long-term rate changes — an important
distinction when interpreting network degree distributions.

In [ ]:
year_counts = df_net["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(18, 5))
sns.barplot(x=year_counts.index, y=year_counts.values, color="steelblue", ax=ax)
ax.set_title("Number of Seismic Events (M ≥ 1.5) in Italy & Seas per Year", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Earthquakes")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_event_counts_italy")
plt.show()

fig, ax = plt.subplots(figsize=(18, 5))
sns.boxenplot(data=df_net, x="year", y="magnitude", palette="viridis", ax=ax)
ax.set_title("Magnitude Distribution per Year", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Magnitude")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_magnitude_dist_italy")
plt.show()

## Magnitude and Depth Analysis

The magnitude histogram should follow the Gutenberg-Richter exponential decay
above the completeness threshold $M_c$; the rapid drop below $M_c$ indicates
under-reporting and should be excluded from b-value fits.

Depth histograms for Italy typically show a **crustal peak** at 5–15 km
(Apennine seismogenic layer) and a secondary cluster at 50–80 km
(Tyrrhenian subducting slab). The magnitude-vs-depth scatter tests whether
larger events occur preferentially at mid-crustal depths, as expected for
brittle-ductile boundary ruptures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df_net["magnitude"].dropna(), bins=np.arange(1.45, 8.55, 0.1),
             edgecolor="black", color="skyblue")
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Magnitude Distribution")

axes[1].hist(df_net["depth_km"].dropna(), bins=100, edgecolor="black", color="salmon")
axes[1].set_xlabel("Depth (km)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Depth Distribution")
plt.tight_layout()
savefig("magnitude_depth_distributions_italy")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df_net["magnitude"], df_net["depth_km"], alpha=0.04, s=2, color="crimson")
axes[0].invert_yaxis()
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Depth (km)")
axes[0].set_title("Magnitude vs Depth")

hb = axes[1].hexbin(df_net["magnitude"], df_net["depth_km"],
                    gridsize=50, cmap="inferno", mincnt=1, bins="log")
plt.colorbar(hb, ax=axes[1], label="log₁₀(count)")
axes[1].invert_yaxis()
axes[1].set_xlabel("Magnitude")
axes[1].set_ylabel("Depth (km)")
axes[1].set_title("Magnitude vs Depth (Density)")
plt.tight_layout()
savefig("magnitude_vs_depth_italy")
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
df_net["mag_bin"] = pd.cut(df_net["magnitude"], bins=[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 8.0])
sns.boxplot(data=df_net, x="mag_bin", y="depth_km", palette="coolwarm",
            showfliers=False, ax=ax)
ax.invert_yaxis()
ax.set_title("Depth Distribution by Magnitude Bin", fontsize=16)
ax.set_xlabel("Magnitude Range")
ax.set_ylabel("Depth (km)")
plt.tight_layout()
savefig("depth_by_magnitude_bin_italy")
plt.show()

## Seismicity Map

Interactive spatial overview of significant events (M ≥ 5) on an Italy basemap.
The $M \geq 5$ threshold retains the most damaging and best-located events while
keeping the map legible; lower thresholds flood the display with micro-seismicity.

Expected pattern for Italy: dense activity along the Apennine arc, Calabrian
subduction zone, and Friuli region — corresponding to the fault systems that
should generate the highest-degree nodes in the Abe-Suzuki network.

In [ ]:
mag_cut  = 5
df_map   = df_net[df_net["magnitude"] >= mag_cut].copy()
print(f"Mapping {len(df_map)} significant earthquakes (M ≥ {mag_cut})...")

fig = px.scatter_map(
    df_map,
    lat="latitude", lon="longitude",
    color="magnitude", size="magnitude",
    color_continuous_scale="matter",
    zoom=4.5, center={"lat": 41.9, "lon": 12.5},
    map_style="carto-positron",
    hover_name="time",
    hover_data={"latitude": ":.3f", "longitude": ":.3f",
                "depth_km": True, "magnitude": True},
    title=f"Interactive Seismic Map: Italy (M ≥ {mag_cut})",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
save_plotly(fig, "seismicity_map_italy")
fig.show()

## Gutenberg-Richter Law

The Gutenberg-Richter relation states $\log_{10} N(\geq M) = a - bM$, where
$N(\geq M)$ is the cumulative count of earthquakes with magnitude $\geq M$,
$a$ (seismicity productivity) and $b \approx 1$ globally (Gutenberg & Richter 1944).

**Sensitivity test**: we vary the upper truncation $M_{\max}$ across
[4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0] to assess fit stability.
A b-value that changes strongly with $M_{\max}$ signals either catalog
incompleteness at small magnitudes or anomalous stress heterogeneity.
Values $b < 0.8$ or $b > 1.3$ warrant physical interpretation
(high-stress asperities vs fluid-saturated fault zones, respectively).

In [ ]:
max_mags   = [4, 4.5, 5, 5.5, 6, 6.5, 7]
plot_flags = [True, False, False, True, False, False, True]

gr_results = [
    fit_gr_law(df_net, max_mag=m, plot=p)
    for m, p in zip(max_mags, plot_flags)
]
df_gr = pd.DataFrame(gr_results).sort_values("max_mag")
display(df_gr)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df_gr["max_mag"], df_gr["a_value"], "o-", color="tab:blue", label="a-value")
ax1.set_xlabel("Max Magnitude Used for Fit")
ax1.set_ylabel("a-value", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(df_gr["max_mag"], df_gr["b_value"], "s-", color="tab:red", label="b-value")
ax2.set_ylabel("b-value", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title("Gutenberg-Richter Parameters vs Max Magnitude")
ax1.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
savefig("gr_parameters_sensitivity_italy")
plt.show()

## Omori Law — Amatrice 2016

The Omori-Utsu law describes aftershock rate decay: $n(t) = K / (c + t)^p$,
where $t$ is time since the mainshock, $p \approx 1$ globally (Utsu et al. 1995),
$K$ scales productivity, and $c$ prevents the singularity at $t = 0$.

The **Amatrice 2016** (M6.2, 24 Aug 2016, central Apennines) sequence is
among the most damaging in recent Italian history. $p > 1$ would indicate
fast aftershock decay typical of warm or fluid-saturated crust;
$p < 1$ suggests a cold, dry fault zone with prolonged seismic hazard.

In [ ]:
print("Fitting Omori law for Amatrice 2016 sequence...")
res_amatrice = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("2016-08-24 01:36:32", utc=True),
    days=60,
    lat_range=(42.0, 43.5),
    lon_range=(12.5, 14.0),
    event_name="Amatrice 2016",
    mag_cut=1.5,
)

## Omori Law — L'Aquila 2009

The **L'Aquila 2009** (M6.3, 6 Apr 2009) sequence produced one of the
best-documented Italian aftershock series (Chiaraluce et al. 2011).
Fitting $n(t) = K / (c + t)^p$ on the 50-day window tests whether
decay is faster or slower than Amatrice — a cross-sequence comparison
that informs the temporal structure of the Abe-Suzuki network edges
originating from the L'Aquila cell cluster.

In [ ]:
print("Fitting Omori law for L'Aquila 2009 sequence...")
res_aquila = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("2009-04-06 01:32:40.4+00:00", utc=True),
    days=50,
    lat_range=(41.5, 43.0),
    lon_range=(12.5, 14.0),
    event_name="L'Aquila 2009",
    mag_cut=1.5,
)

df_omori = pd.DataFrame([
    {"event": "Amatrice 2016", **res_amatrice},
    {"event": "L'Aquila 2009", **res_aquila},
])
print("\nOmori fit comparison:")
display(df_omori)